If you've never done it before, or even if you've only done it a few times, pre-registration can feel intimidating.

* What if your pre-registration is not detailed enough? 
* What if you forget something?
* What if your design is complex, and you pre-register the wrong analysis?
* What if you pre-register 400 participants, but then a pandemic hit, all the labs around the world are closed, and you only get 300 responses? (Complete hypothetical scenario, never happened to me before).

While these are valid concerns, your conclusion should never be "I won't pre-register this study". Why? Because **even a "bad" pre-registration is dramatically better than no pre-registration.**

Let's see how.

## First, a Recap: The Benefits of Pre-Registration

There are two main benefits to pre-registration:

### It helps us (researchers) stick to our planned analysis, and generate fewer false-positives.

Let's assume a researcher who ran a study to test the hypothesis that anthropomorphic potatoes are perceived as as more "friendly" than anthropomorphic asparagus. Without a pre-registration, it can be tempting to decide, after seeing the data, that *actually* the effect will only work on men. Or that "warm" is a better measure of friendliness than "friendly", at least for a vegetable. Or that they should exclude the 10% of participants who failed an attention check. Or that they should control for the agreeableness of the participants. Or that since the effect is encouragingly trending in the right direction, they should add a few participants to be sure. You get the idea: A pre-registration keeps our ["researchers' degrees of freedom"](https://journals.sagepub.com/doi/pdf/10.1177/0956797611417632) in check. It encourages researchers to stick to their original idea of how the data should be collected, transformed, and analyzed, which minimizes the odds of false-positive results.

### It allows others (reviewers and the readers) to interpret the p-values contained in a paper.

A p-value loses its meaning when the null hypothesis is not defined a priori. If a study is not pre-registered, the reader cannot tell if a p-value reported in the paper is the smallest out of 20 different tests that the researchers ran, or the output of the single test that the researchers had planned to run before seeing any data. A pre-registration allows the readers to determine how "severe" the statistical test was, and therefore how much evidence against the null the p-value actually provides.

Now, why would a "bad" pre-registration (i.e., one that is incomplete, or incorrect, such that one would deviate from it when analyzing the data) be so much better than no pre-registration?

## Combinatorics, That's Why!

Let's go back to our researcher studying the friendliness of anthropomorphic vegetables. We will assume that this researcher has the following degrees of freedom available to them:
* How many participants to collect: 50 per cell, or 100 per cell.
* Which participants to include in the analysis: Men only, women only, or both.
* Which dependent variable to use: "Friendly" or "Warm" (these variables are correlated at r = .5).
* Whether to exclude the 10% of participants who will fail the attention check.
* Whether to control for the agreeableness of the participant.

In total, the researcher has 3 x 2 x 2 x 2 x 2 = 48 different possible specifications that they can run. 

Now, imagine that this researcher decides to write an extremely minimal pre-registration: The only thing that the pre-registration states is "We will collect 50 participants per cell, for a total of 100 participants." That's it. 

It's a very bad pre-registration, one that addresses only *one* of the many degrees of freedom available to the researcher. And yet, by restricting a *single* degree of freedom, the number of possible specifications is *halved*: The researcher can now only run 3 x 2 x 2 x 2 = 24 specifications.

This is because degrees of freedom are combinatorial: They do not add up, they get multiplied. For this reason, the benefits of controlling for degrees of freedom are not linear: Dropping just one or two degrees of freedom can have a dramatic impact on the number of specifications that the researcher can run.

How would that actually affect the researchers' likelihood of finding a false-positive?

## Simulating Degrees of Freedom

In [1]:
import multiprocessing

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from joblib import Parallel, delayed
from IPython.display import display

np.random.seed(239013280)


def hist_no_edge(x, *args, **kwargs):
    """
    Plot a histogram without the left and right edges. Useful for survival curves.
    """
    bins = kwargs.pop("bins")
    cumulative = kwargs.pop("cumulative", False)
    # Add a bin for all p-values between the upper bin and 1.
    cnt, edges = np.histogram(x, bins=bins.tolist() + [1])
    ax = plt.gca()
    if cumulative:
        prop = cnt.cumsum() / cnt.sum()
        ax.step(edges[:-2], prop[:-1], *args, **kwargs)
    else:
        ax.step(edges[:-2], cnt, *args, **kwargs)


def plot_simulation_results(data):
    """
    Plot the survival curve of all the simulations.
    """
    gb = data.groupby(["Pre-Reg"])  # Data by group.

    # Type I error at various alpha levels
    p_05 = gb["$\\alpha_{{05}}$"].mean()
    p_01 = gb["$\\alpha_{{01}}$"].mean()
    p_001 = gb["$\\alpha_{{001}}$"].mean()

    # Labels for lines
    levels = ["None", "Bad", "Quasi-Perfect", "Perfect"]
    labels = ["None", "Bad", "Quasi-Perfect", "Perfect"]
    maxcoord = 4

    ycoords = [0.65, 0.71, 0.77, 0.83][::-1]  # Coordinates to plot legend
    pal = sns.color_palette()[0:4]  # Colors
    pal = [pal[3], pal[1], pal[2], pal[0]]
    # Initialize the plot
    g = sns.FacetGrid(
        data,
        hue="Pre-Reg",
        hue_order=levels,
        height=5,
        aspect=1.2,
        sharey=True,
        sharex=True,
        hue_kws=dict(ls=[(0, ()), (0, ()), (1, (5, 5)), (1, ())]),
        palette=pal,
    )

    # Map the survival curve to each panel and color
    g.map(
        hist_no_edge,
        "pvalue",
        cumulative=True,
        bins=np.logspace(np.log10(10e-40), np.log10(0.05), 10000),
        lw=1.5,
    )

    # Change the axes labels
    g.set_ylabels("Fraction of Significant Experiments ($p < \\alpha$)")
    g.set_xlabels("Significance Threshold ($\\alpha$)")
    ax = g.ax

    ax.set_xlim(10e-8, 0.05)
    ax.set_xscale("log")
    ax.set_xticks([10e-8, 10e-7, 10e-6, 10e-5, 10e-4, 10e-3, 0.05])
    ax.set_xticklabels(["10e-8", "10e-7", ".00001", ".0001", ".001", ".01", ".05"])
    ax.set_xticks([], minor=True)
    ax.invert_xaxis()
    ax.annotate(
        f"Type I Error Rate at $[\\alpha_{{05}}, \\alpha_{{01}}, \\alpha_{{001}}]$",
        (1, 0.89),
        color="black",
        fontsize=10,
        ha="right",
        xycoords="axes fraction",
    )
    ax.set_yticks(np.arange(0, 0.5, 0.05))
    # Add the false-positive annotations
    for k, (lab, lev) in enumerate(zip(labels, levels)):
        ax.annotate(
            f"{lab}: [{p_05.loc[lev]:.2f}, "
            f" {p_01.loc[lev]:.2f}, "
            f"{p_001.loc[lev]:.3f}]",
            (1, ycoords[k]),
            color=pal[k],
            ha="right",
            xycoords="axes fraction",
            fontsize=10,
        )


def test_specifications_and_get_pvals(seed):
    """
    A function that will be parallelized for faster results.
    """
    np.random.seed(seed)
    gender = np.random.choice([0, 1], (200))  # Gender of participants
    dvar = np.random.multivariate_normal(
        mean=[0, 0],
        cov=[[1, 0.5], [0.5, 1]],
        size=(200),  # Two dependent variables, correlated at .5
    )
    ivar = np.random.choice([0, 1], (200))  # Independent variable.
    excl = np.random.choice([0, 1], p=[0.9, 0.1], size=(200))  # Failing attention check
    agg = np.random.normal(0, 1, size=(200))  # Agreeableness scores
    pvs = np.empty(
        shape=(3, 2, 2, 2, 2)
    )  # Vector to store p-values of 48 possible specifications.
    df = pd.DataFrame(
        {
            "dv1": dvar[:, 0],
            "dv2": dvar[:, 1],
            "iv": ivar,
            "excl": excl,
            "agg": agg,
            "gender": gender,
        }
    )  # Dataset
    # We iterate over all possible combinations of specifications...
    for i, g_select in enumerate([[0], [1], [0, 1]]):
        for j, dv_select in enumerate(["dv1", "dv2"]):
            for k, excl_select in enumerate([False, True]):
                for l, agg_control in enumerate([False, True]):
                    for m, ssize in enumerate([100, 200]):
                        d = df.iloc[0:ssize].query("(gender in @g_select)")
                        if excl_select:
                            d = d.query("excl == 0")
                        pvs[i, j, k, l, m] = (
                            (smf.ols(f"{dv_select}~iv+agg", data=d).fit().pvalues[1])
                            if agg_control
                            else (smf.ols(f"{dv_select}~iv", data=d).fit().pvalues[1])
                        )
    return pvs

To answer this question, I used one of my favorite tools: Simulations!

I generated a large number (10,000) of simulated experiments testing the friendliness of anthropomorphic vegetables. In these datasets, the null hypothesis is true: People find anthropomorphic potatoes and anthropomorphic asparagus equally friendly ([but what do I know, maybe they don't!](https://veggietales-the-ultimate-veggiepedia.fandom.com/wiki/Archibald_Asparagus)).

For each dataset, I calculated the **smallest p-value that the researcher would obtain** under four different constraints:
* No pre-registration: The researchers is free to try all 48 of the specifications described earlier.
* A bad pre-registration: The researcher is constrained to run 100 participants, but is free to try all the other 24 specifications.
* A quasi-perfect pre-registration: The researcher pre-registered almost everything, but forgot to specify whether they'll exclude the participants who failed the attention check, such they can try 2 specifications.
* A perfect pre-registration: The researchers can only run the 1 specification they pre-registered.

The graph below shows the distribution of the p-values that these procedures will generate.

In [2]:
n_boots = 10000
seeds = np.random.randint(
    0, 2147483647, size=n_boots
)  # Seeds sent to parallelized functions
num_cores = multiprocessing.cpu_count()  # Number of cores
# Parallelize simulations across cores
results = np.array(
    Parallel(n_jobs=num_cores)(
        delayed(test_specifications_and_get_pvals)(s) for s in seeds
    )
)

# No Pre-Reg: Smallest p-value observed across 48 specifications
results_baseline = np.min(np.reshape(results, (n_boots, -1)), axis=1)

# Bad Pre-Reg: Smallest p-value observed across 24 specifications
results_restricted = np.min(
    np.reshape(results[:, :, :, :, :, 0], (n_boots, -1)), axis=1
)

# Quasi-Perfect Pre-Reg: Smallest p-value observed across two specifications
results_quasiperfect = np.min(
    np.reshape(results[:, 2, 0, :, 0, 0], (n_boots, -1)), axis=1
)

# Perfect Pre-Reg: p-value observed in the only specification
results_perfect = np.min(np.reshape(results[:, 2, 0, 0, 0, 0], (n_boots, -1)), axis=1)

# All results together in a dataframe
results_all = np.hstack(
    [
        results_baseline,
        results_restricted,
        results_quasiperfect,
        results_perfect,
    ]
)
df_results = pd.DataFrame(
    {
        "pvalue": results_all,
        "Pre-Reg": np.repeat(["None", "Bad", "Quasi-Perfect", "Perfect"], n_boots),
    }
)

# Computing the Type I error rates at .05, 01, and .001:
df_results["$\\alpha_{{05}}$"] = (df_results.pvalue < 0.05) * 1
df_results["$\\alpha_{{01}}$"] = (df_results.pvalue < 0.01) * 1
df_results["$\\alpha_{{001}}$"] = (df_results.pvalue < 0.001) * 1

# Plotting the results
plot_simulation_results(df_results)
fig = plt.gcf()
plt.close()
display(fig, metadata=dict(filename="Fig1"))

Exception in thread ExecutorManagerThread:
Traceback (most recent call last):
  File "/home/andreq/anaconda3/envs/research2/lib/python3.9/threading.py", line 973, in _bootstrap_inner
    self.run()
  File "/home/andreq/anaconda3/envs/research2/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py", line 575, in run
    self.flag_executor_shutting_down()
  File "/home/andreq/anaconda3/envs/research2/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py", line 770, in flag_executor_shutting_down
    self.kill_workers(reason="executor shutting down")
  File "/home/andreq/anaconda3/envs/research2/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py", line 781, in kill_workers
    recursive_terminate(p)
  File "/home/andreq/anaconda3/envs/research2/lib/python3.9/site-packages/joblib/externals/loky/backend/utils.py", line 28, in recursive_terminate
    _recursive_terminate_without_psutil(process)
  File "/home/andreq/anaconda3/envs/research2/l

The lines that you see in this figure are called survival curves: They show the percentage of experiments that produced a significant result (on the y-axis) at a given significance threshold (on the x-axis). 

We first see that when we have a perfect pre-registration (the blue line), the false-positive rates are nominal:
* 5% of the experiments yield a significant result at α = .05
* 1% at α = .01
* .1% at α = .001

This isn't surprising: As discussed earlier, the main benefit of pre-registration is to maintain false-positive rates at their nominal level.

We also see that when there is no pregistration (the red line), the false-positive rates jump to unacceptable levels: 
* 40% of the experiments yield a significant result at α = .05
* 12% at α = .01
* .15% at α = .001.

This isn't, again, a surprising result: We know that many uncontrolled researchers' degrees of freedom lead to inflated false-positive rates.

The interesting result is what happens with our "bad" pre-registration (the orange line), which is just removing a single degrees of freedom: It dramatically drops the false-positive rates!
* 27% of the experiments yield a significant result at α = .05 (a 33% drop)
* 7% at α = .01 (a 40% drop)
* .09% at α = .001 (a 43% drop)

These false-positive rates are still unacceptably high: It is, after all, a terrible pre-registration... But they are much lower than the alternative of not pre-registering.

Finally, we see that our imperfect pre-registration (leaving only one degree of freedom to the researcher) achieves false-positive rates that are elevated, but very close to those of a perfect pre-registration. This should be an encouraging result for people who are afraid of making mistakes in their pre-registration!

## Concluding Thoughts

For authors:

* Don't be afraid to pre-register. It can feel intimidating if you've never tried it, but there are [excellent ressources available explaining how to write good pre-registrations](http://datacolada.org/64), and the platform [aspredicted.org](aspredicted.org) makes it really easy.
* Even if you fear you will make mistakes, or think you will deviate from your pre-registration, you should still pre-register your studies: The improvement in false-positive rates you'll achieve is worth it.

For reviewers and readers:

* If a study is not pre-registered, you cannot know how many degrees of freedom were exerted by the researchers: Maybe none, maybe many. As a reader, this should lead you to exert caution when interpreting the p-values, [particularly if they are between .01 and .05](http://datacolada.org/45). As a reviewer, it might be a cue to ask for a pre-registered replication of the finding.
* Not all pre-registrations are equal. Reading "the study was pre-registered" doesn't tell you how many researchers' degrees of freedom were controlled for. This can only be assessed by carefully reading the pre-registration, and comparing it to the design of the study.
* Don't automatically assume that any deviation from the pre-registration means a false-positive result. It is a counter-productive attitude that can discourage people from pre-registering. As shown in the simulations above, deviations can have a minor impact on false-positive rates as long as 1) the original pre-registration controlled for most (ideally all) the researchers' degrees of freedom and 2) the deviation(s) are small and limited in scope. 
* On the contrary, deviations should raise eyebrows 1) when there are many of them, 2) when the original pre-registration was vague, and 3) when the p-value is large (again, between .01 or .05).